# FloodScout — train the flood model

Trains the model FloodScout scores regions with, on Google Colab.

**What it learns from.** One row per Kenyan county per month, 1996–2021: rainfall for the 30 days ending that month (ERA5 reanalysis, at one point or averaged over the county), the El Niño and Indian Ocean Dipole indices (NOAA ONI and DMI), elevation, slope and county area, and whether the Dartmouth Flood Observatory recorded a large flood there in that window. Years before 1996 are left out because the archive barely recorded Kenyan floods then.

**What it outputs.** The probability that a large flood is recorded in a region in a 30-day window.

The training code is not in this notebook. It is `riskengine/ml/train.py` from the repository, packed in the bundle below, so Colab and the app always run the same code.

**Before you start**, commit your work and build the bundle on your machine from the repository root:

```
docker compose run --rm web python manage.py build_training_set
docker compose run --rm web python manage.py build_colab_bundle
```

The second command refuses to run with uncommitted changes. It writes `data/processed/colab_bundle_<commit>.tar.gz`, holding the committed code, the training set, a `COMMIT` file and the SHA-256 of the training set. Put that file in Google Drive under `MyDrive/FloodScout/data/`.

**What a run leaves behind.** A folder `MyDrive/FloodScout/runs/<run id>/` with the model, `metrics.json`, every chart, the installed packages (`environment.txt`), the training log (`log.txt`) and `run.json`, which names the commit and the bundle. The run id is the UTC start time and the first seven characters of the commit.

## 1. Match the app's scikit-learn

The saved model is a pickle. It only loads reliably under the scikit-learn version that wrote it, so this must match `requirements.txt`. If Colab asks to restart the session after this cell, restart and carry on from step 2.

In [ ]:
%pip install -q "scikit-learn==1.9.1"
import sklearn
print(sklearn.__version__)

## 2. Mount Google Drive and unpack the bundle

Colab's own disk is wiped when the session ends, so the bundle is read from Google Drive and the results are written back to it. The newest `colab_bundle_<commit>.tar.gz` in `MyDrive/FloodScout/data/` is used; set `BUNDLE` by hand to rerun an older one.

In [ ]:
import datetime
import hashlib
import json
import os
import shutil
import subprocess
import tarfile
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/FloodScout")
DATA_DIR = DRIVE_ROOT / "data"
RUNS_DIR = DRIVE_ROOT / "runs"
CODE_DIR = Path("/content/floodscout")

bundles = sorted(DATA_DIR.glob("colab_bundle_*.tar.gz"), key=lambda path: path.stat().st_mtime)
assert bundles, f"Put colab_bundle_<commit>.tar.gz in {DATA_DIR}"
BUNDLE = bundles[-1]  # the newest; or set explicitly

shutil.rmtree(CODE_DIR, ignore_errors=True)  # nothing left over from an earlier bundle
with tarfile.open(BUNDLE) as bundle:
    bundle.extractall(CODE_DIR, filter="data")
os.chdir(CODE_DIR)

TRAINING_SET = Path("data/processed/training_set.csv")
COMMIT = Path("COMMIT").read_text().strip()
DATASET_SHA256 = hashlib.sha256(TRAINING_SET.read_bytes()).hexdigest()
assert DATASET_SHA256 == Path("DATASET_SHA256").read_text().strip(), "The training set does not match the bundle's record"

RUN_ID = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%dT%H%MZ") + "_" + COMMIT[:7]
RUN_DIR = RUNS_DIR / RUN_ID
(RUN_DIR / "figures").mkdir(parents=True, exist_ok=True)
os.environ["FLOODSCOUT_GIT_COMMIT"] = COMMIT  # read by train.git_commit()
os.environ["FLOODSCOUT_RUN_ID"] = RUN_ID  # recorded in metrics.json
print("bundle", BUNDLE.name)
print("commit", COMMIT)
print("training set sha256", DATASET_SHA256)
print("run", RUN_ID)

## 3. What the data looks like

One row is one county in one month. Floods are rare, about one row in ten, which is why the models are judged on average precision and not on accuracy: always answering "no flood" would be right nine times in ten and useless.

The second chart is the final test split. The years from 2017 on are set aside before any model is compared.

In [ ]:
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

sys.path.insert(0, ".")
from riskengine.ml import train

MUTED, GRID, AXIS = "#898781", "#e1e0d9", "#c3c2b7"
COLOURS = {
    "logreg": "#2a78d6",
    "random-forest": "#eb6834",
    "gradient-boosting": "#1baf7a",
    "season-only": MUTED,
    "base-rate": AXIS,
}
LABELS = {
    "logreg": "Logistic Regression",
    "random-forest": "Random Forest",
    "gradient-boosting": "Gradient Boosting",
    "season-only": "Season only",
    "base-rate": "Base rate",
}
BANDS = ["Low", "Moderate", "High", "Very High"]
plt.rcParams.update({
    "figure.dpi": 150,
    "figure.facecolor": "white",
    "font.size": 9,
    "axes.titlesize": 10,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": AXIS,
    "axes.axisbelow": True,
    "axes.grid": True,
    "grid.color": GRID,
    "grid.linewidth": 0.6,
    "xtick.color": MUTED,
    "ytick.color": MUTED,
    "legend.frameon": False,
})


def show(fig, name):
    """Save the chart in the run folder in Drive, then show it."""
    fig.savefig(RUN_DIR / "figures" / f"{name}.png", dpi=150)
    plt.show()


rows = pd.read_csv(TRAINING_SET, parse_dates=["date"])
rows = rows[rows["date"].dt.year >= train.FIRST_TRAINING_YEAR]
by_year = rows.groupby(rows["date"].dt.year)["flood"].sum()

fig, (left, right) = plt.subplots(1, 2, figsize=(10, 3.2), gridspec_kw={"width_ratios": [1, 2.6]})

counts = [int((rows["flood"] == 0).sum()), int((rows["flood"] == 1).sum())]
bars = left.bar(["No flood recorded", "Flood recorded"], counts, color=[AXIS, COLOURS["logreg"]], width=0.6)
left.bar_label(bars, [f"{count:,}\n({count / len(rows):.1%})" for count in counts], padding=3)
left.set_ylim(0, max(counts) * 1.2)
left.set_title("County-months by label")
left.grid(axis="x", visible=False)

is_test = by_year.index >= train.FINAL_TEST_FROM
right.bar(by_year.index[~is_test], by_year[~is_test], color=COLOURS["logreg"], width=0.8,
          label=f"Development, {by_year.index.min()} to {train.FINAL_TEST_FROM - 1}")
right.bar(by_year.index[is_test], by_year[is_test], color=COLOURS["random-forest"], width=0.8,
          label=f"Final test, {train.FINAL_TEST_FROM} to {by_year.index.max()}")
right.set_xticks(by_year.index[::2])
right.set_title("County-months with a recorded flood, by year")
right.grid(axis="x", visible=False)
right.legend(loc="upper right")
fig.tight_layout()
show(fig, "01_data_overview")

## 4. Train

Compares Logistic Regression, Random Forest and Gradient Boosting, holding out whole years at a time, against two baselines: always guessing the base rate, and a model that knows only the month. Each model's settings are searched over a small grid with `GridSearchCV` on the same year folds, and the searched settings are used only where they beat the fixed ones on held-out years. Then the best model is refitted on all the data and its probabilities are calibrated if that helps on held-out years.

The whole procedure runs twice, once on every year and once on 1996 to 2016 for the final test, so this takes a while on Colab's two processors.

Nothing is printed until it finishes: the output is collected so it can be saved as `log.txt` in the run folder, with how long the training took.

In [ ]:
import contextlib
import io
import time

from geodata.services.climate_indices import fetch_dmi, fetch_oni

log = io.StringIO()
started = time.perf_counter()
with contextlib.redirect_stdout(log):
    data = train.load_training_set(TRAINING_SET)
    chosen = train.select(data)  # kept for the curves in step 5
    artifact, metrics = train.build(data, chosen, TRAINING_SET, fetch_oni(), fetch_dmi())
    train.save(artifact, metrics)
    print("Trained", artifact["version"])
TRAIN_SECONDS = round(time.perf_counter() - started, 1)
print(f"Training took {TRAIN_SECONDS} seconds", file=log)
(RUN_DIR / "log.txt").write_text(log.getvalue())
print(log.getvalue(), end="")

## 5. How good is it?

All figures are on years the model did not see. **Average precision** is the one the winner is picked on. Compare it with both baselines: beating the base rate shows the model knows *something*; beating the season-only model shows rainfall, the ocean indices and terrain add something beyond "it is the rainy season". **Brier** is lower-is-better.

In [ ]:
print(f"{metrics['rows']} county-months from {metrics['period'][0][:4]}, {metrics['positives']} with a recorded flood "
      f"(base rate {metrics['base_rate']:.1%}); rainfall: {metrics['rain_source']}")
print(f"Selected {metrics['selected']}: average precision {metrics['lift_over_base_rate']}x the base rate, "
      f"{metrics['lift_over_season_only']}x the season-only model. Calibration kept: {metrics['calibration']}")
comparison = pd.DataFrame(metrics["comparison"]).T
for name, scores in metrics["baselines"].items():
    comparison.loc[f"({name})"] = pd.Series(scores)
comparison

The same comparison as a chart. The grey bars are the two baselines.

In [ ]:
scores = {**metrics["comparison"], **metrics["baselines"]}
fig, axes = plt.subplots(1, 2, figsize=(10, 2.8), sharey=True)
for ax, (metric, title) in zip(axes, [("average_precision", "Average precision"), ("roc_auc", "ROC AUC")]):
    order = sorted(scores, key=lambda name: scores[name]["average_precision"])
    bars = ax.barh(
        [LABELS[name] + (" (selected)" if name == metrics["selected"] else "") for name in order],
        [scores[name][metric] for name in order],
        color=[COLOURS[name] for name in order],
        height=0.6,
    )
    ax.bar_label(bars, fmt="%.4f", padding=3)
    ax.set_xlim(0, max(scores[name][metric] for name in order) * 1.15)
    ax.set_title(f"{title} on held-out years")
    ax.grid(axis="y", visible=False)
fig.tight_layout()
show(fig, "02_model_comparison")

The settings search. `fixed` is each model at the settings written in `train.py`, `searched` at the settings `GridSearchCV` chose from the grid, both scored on held-out years with the county normals refitted per fold. `adopted` says whether the searched settings replaced the fixed ones. The same table for the final test's 1996 to 2016 is underneath: the search is repeated there without the test years.

In [ ]:
def tuning_table(tuning):
    return pd.DataFrame({
        LABELS[name]: {
            "grid": record["grid"],
            "fixed settings": record["fixed_settings"],
            "searched settings": record["searched_settings"],
            "fixed AP": record["fixed"]["average_precision"],
            "searched AP": record["searched"]["average_precision"],
            "adopted": record["adopted"],
        }
        for name, record in tuning.items()
    }).T

with pd.option_context("display.max_colwidth", None):
    display(tuning_table(metrics["tuning"]))
    display(tuning_table(metrics["final_test"]["tuning"]))

The curves behind those two numbers, from each row's probability under a model that never saw its year.

**ROC**: how many floods are caught against how many non-floods are wrongly flagged, as the warning threshold moves. The diagonal is guessing. **Precision-recall**: of the warnings given, how many were floods, against how many floods were caught. The dashed line is the base rate.

In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve

held_out = {**chosen.held_out, "season-only": chosen.season}

fig, (roc, pr) = plt.subplots(1, 2, figsize=(10, 4))
for name, probabilities in held_out.items():
    style = {"color": COLOURS[name], "label": LABELS[name], "linewidth": 2 if name == chosen.best else 1.3}
    false_alarms, caught, _ = roc_curve(data.y, probabilities)
    roc.plot(false_alarms, caught, **style)
    precision, recall, _ = precision_recall_curve(data.y, probabilities)
    pr.plot(recall, precision, **style)
roc.plot([0, 1], [0, 1], color=AXIS, linestyle="--", linewidth=1, label="Guessing")
roc.set_xlabel("Non-floods wrongly flagged (false positive rate)")
roc.set_ylabel("Floods caught (recall)")
roc.set_title("ROC curve, held-out years")
roc.legend(loc="lower right")
pr.axhline(data.y.mean(), color=AXIS, linestyle="--", linewidth=1, label="Base rate")
pr.set_xlabel("Floods caught (recall)")
pr.set_ylabel("Warnings that were floods (precision)")
pr.set_ylim(0, 0.6)  # the first few rows jump to 1.0; the comparison is below that
pr.set_title("Precision-recall curve, held-out years")
pr.legend(loc="upper right")
fig.tight_layout()
show(fig, "03_roc_pr_curves")

How often a flood was actually recorded in each risk band, on held-out years, after calibration. The bands are only useful if this rises from Low to Very High, and `mean_predicted` should sit close to `observed_flood_rate`.

In [ ]:
pd.DataFrame(metrics["band_calibration"]).set_index("band")

Bars are the share of county-months in each band that had a recorded flood; the dot is what the model said on average. Left is cross-validation, right is the 2017 to 2021 final test. The bands are useful if the bars rise from Low to Very High, and trustworthy as probabilities only if the dots sit on top of the bars.

In [ ]:
panels = [
    (f"Cross-validation, all years ({LABELS[metrics['selected']]})", metrics["band_calibration"]),
    (f"Final test, 2017 to 2021 ({LABELS[metrics['final_test']['selected']]})", metrics["final_test"]["band_calibration"]),
]
fig, axes = plt.subplots(1, 2, figsize=(10, 3.4), sharey=True)
for ax, (title, bands) in zip(axes, panels):
    observed = [100 * (band["observed_flood_rate"] or 0) for band in bands]
    predicted = [100 * (band["mean_predicted"] or 0) for band in bands]
    bars = ax.bar(BANDS, observed, color=COLOURS["logreg"], width=0.6, label="Floods recorded")
    ax.bar_label(bars, fmt="%.1f%%", padding=3)
    ax.scatter(BANDS, predicted, color="#0b0b0b", s=36, zorder=3, label="Model's average probability")
    ax.set_title(title)
    ax.grid(axis="x", visible=False)
    ax.yaxis.set_major_formatter("{x:.0f}%")
axes[0].legend(loc="upper left")
fig.tight_layout()
show(fig, "04_risk_bands")

Calibration on held-out years. ROC AUC barely moves (calibration does not change the ranking); the one with the lowest Brier is kept.

In [ ]:
pd.DataFrame(metrics["calibration_comparison"]).T

Average precision in each fold, with the years it held out. The pooled figures above hide how much the result depends on which years are held out; the spread here is the honest error bar.

In [ ]:
folds = pd.DataFrame(
    {name: scores["average_precision"] for name, scores in fold["scores"].items()}
    | {"years": ", ".join(map(str, fold["years"])), "floods": fold["positives"]}
    for fold in metrics["folds"]
)
display(folds)
pd.DataFrame(metrics["fold_summary"]).T

The same per-fold scores as a chart, and which years each fold holds out. A fold is always whole years, never a random sample of rows.

In [ ]:
fold_names = [f"Fold {index + 1}" for index in range(len(metrics["folds"]))]
models = [*metrics["comparison"], "season-only"]
fig, ax = plt.subplots(figsize=(10, 3.2))
width = 0.8 / len(models)
for offset, name in enumerate(models):
    ax.bar(
        np.arange(len(fold_names)) + (offset - (len(models) - 1) / 2) * width,
        [fold["scores"][name]["average_precision"] or 0 for fold in metrics["folds"]],
        width=width * 0.9,
        color=COLOURS[name],
        label=LABELS[name],
    )
ax.set_xticks(np.arange(len(fold_names)), fold_names)
ax.set_title("Average precision in each held-out fold")
ax.grid(axis="x", visible=False)
ax.legend(ncols=len(models), loc="upper center", bbox_to_anchor=(0.5, -0.12))
fig.tight_layout()
show(fig, "05_fold_scores")

FOLD_COLOURS = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300"]
fold_of = {year: index for index, fold in enumerate(metrics["folds"]) for year in fold["years"]}
years = sorted(fold_of)
fig, ax = plt.subplots(figsize=(10, 1.5))
for year in years:
    ax.bar(year, 1, bottom=1.2, width=0.9, color=FOLD_COLOURS[fold_of[year] % len(FOLD_COLOURS)])
    ax.text(year, 1.7, str(fold_of[year] + 1), ha="center", va="center", color="white", fontweight="bold")
    final = year >= metrics["final_test"]["test_from"]
    ax.bar(year, 1, bottom=0, width=0.9, color=COLOURS["random-forest"] if final else AXIS)
    ax.text(year, 0.5, "T" if final else "D", ha="center", va="center", color="white" if final else "#52514e")
ax.set_xticks(years[::2])
ax.set_yticks([0.5, 1.7], ["Final test split\n(D development, T test)", "Fold that holds\nthe year out"])
ax.tick_params(length=0)
ax.grid(visible=False)
ax.spines[["left", "bottom"]].set_visible(False)
ax.set_title("How the years are split")
fig.tight_layout()
show(fig, "06_year_split")

What happens if a probability above each band edge is treated as a warning. A false negative is a recorded flood with no warning.

In [ ]:
pd.DataFrame(metrics["threshold_errors"]).set_index("threshold")

The 10% row of that table as a confusion matrix: a High or Very High band counts as a warning.

In [ ]:
errors = next(row for row in metrics["threshold_errors"] if row["threshold"] == 0.10)
matrix = np.array([
    [errors["true_positives"], errors["false_positives"]],
    [errors["false_negatives"], errors["true_negatives"]],
])
names = [["Floods caught", "False alarms"], ["Floods missed", "Correctly no warning"]]
fig, ax = plt.subplots(figsize=(5.2, 3.6))
ax.imshow(matrix, cmap="Blues", vmin=0, vmax=matrix.max() * 1.25)
for row in range(2):
    for column in range(2):
        ax.text(
            column, row, f"{matrix[row, column]:,}\n{names[row][column]}",
            ha="center", va="center",
            color="white" if matrix[row, column] > matrix.max() * 0.6 else "#0b0b0b",
        )
ax.set_xticks([0, 1], ["Flood recorded", "No flood recorded"])
ax.set_yticks([0, 1], ["Warning", "No warning"])
ax.xaxis.tick_top()
ax.tick_params(length=0)
ax.grid(visible=False)
ax.spines[:].set_visible(False)
ax.set_xlabel(
    f"Recall {errors['recall']:.1%}, precision {errors['precision']:.1%}, specificity {errors['specificity']:.1%}"
)
fig.tight_layout()
show(fig, "07_confusion_matrix")

Where those errors fall. Left: recorded floods in each calendar month, split into caught and missed. Right: the fifteen counties with the most recorded floods, split the same way.

In [ ]:
breakdown = metrics["error_breakdown"]
months = pd.DataFrame(breakdown["by_month"]).set_index("month")
counties = pd.DataFrame(breakdown["by_county"]).set_index("county").sort_values("floods").tail(15)
MONTHS = ["Jan", "Feb", "Mar", "Apr", "May", "Jun", "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"]

fig, (left, right) = plt.subplots(1, 2, figsize=(10, 4.2), gridspec_kw={"width_ratios": [1.3, 1]})
left.bar(MONTHS, months["caught"], color=COLOURS["logreg"], width=0.7, label="Caught")
left.bar(MONTHS, months["missed"], bottom=months["caught"], color=COLOURS["random-forest"], width=0.7, label="Missed")
for position, (caught, missed) in enumerate(zip(months["caught"], months["missed"])):
    if caught + missed:
        left.text(position, caught + missed + 3, f"{missed / (caught + missed):.0%}", ha="center", color=MUTED)
left.set_title(f"Recorded floods by month (label: share missed), warning above {breakdown['threshold']:.0%}")
left.grid(axis="x", visible=False)
left.legend(loc="upper left")
right.barh(counties.index, counties["caught"], color=COLOURS["logreg"], height=0.7)
right.barh(counties.index, counties["missed"], left=counties["caught"], color=COLOURS["random-forest"], height=0.7)
right.set_title("Recorded floods by county, top 15")
right.grid(axis="y", visible=False)
fig.tight_layout()
show(fig, "08_error_breakdown")

print("False alarms by month:", dict(zip(MONTHS, months["false_alarms"])))

The final test. The years from 2017 on were set aside, the model and calibration were chosen again on 1996 to 2016 alone, and 2017 to 2021 were scored once. This is the figure to trust most, and it is the least flattering. The first row is the model that choice picked; the others are shown for the record.

In [ ]:
test = metrics["final_test"]
print(f"{test['rows']} county-months, {test['positives']} floods, {test['test_years']}")
display(pd.DataFrame(
    {f"{test['selected']} (chosen)": test["scores"]} | test["other_candidates"] | test["baselines"]
).T)
pd.DataFrame(test["band_calibration"]).set_index("band")

The commit, run, data and seed this result came from, as stored in `metrics.json`. `git_commit` is the commit the bundle was built from, read from the bundle's `COMMIT` file, and `training_set_sha256` is the same value the bundle recorded.

In [ ]:
assert metrics["provenance"]["git_commit"] == COMMIT
assert metrics["provenance"]["training_set_sha256"] == DATASET_SHA256
metrics["provenance"]

What the model leans on. Each input is shuffled in turn on the held-out years, and the bar is how far average precision falls, averaged over the six folds (the line is one standard deviation across folds). An input the model does not need scores about zero. Inputs that carry the same information share the credit: shuffling `rain_30d` leaves `rain_30d_ratio` intact. `log_area` is a control for how the labels are made, not a cause of flooding; see `riskengine/ml/README.md`.

In [ ]:
importance = pd.DataFrame(metrics["permutation_importance"]).T.sort_values("mean")
fig, ax = plt.subplots(figsize=(7, 3.8))
bars = ax.barh(importance.index, importance["mean"], xerr=importance["sd"], color=COLOURS[metrics["selected"]],
               height=0.65, error_kw={"ecolor": "#52514e", "elinewidth": 1})
ax.axvline(0, color=AXIS, linewidth=1)
ax.set_title(f"Permutation importance, {LABELS[metrics['selected']]}")
ax.set_xlabel("Fall in held-out average precision when the input is shuffled")
ax.grid(axis="y", visible=False)
fig.tight_layout()
show(fig, "09_permutation_importance")

In which direction. Partial dependence of the saved model on its four most important inputs: the average predicted probability as one input is moved across its range and the others are left as recorded. The ticks along the bottom show where the data is.

In [ ]:
from sklearn.inspection import PartialDependenceDisplay

model = getattr(artifact["pipeline"], "pipeline", artifact["pipeline"])  # under any calibration
top = list(importance.index[::-1][:4])
fig, axes = plt.subplots(1, 4, figsize=(11, 2.8), sharey=True)
PartialDependenceDisplay.from_estimator(
    model,
    train.with_normals(data, artifact["normals"]),
    [artifact["features"].index(name) for name in top],
    feature_names=artifact["features"],
    grid_resolution=30,
    ax=axes,
    line_kw={"color": COLOURS[metrics["selected"]], "linewidth": 2},
)
for ax in axes:
    ax.set_ylabel("")
axes[0].set_ylabel("Average predicted probability")
fig.suptitle("Partial dependence on the four most important inputs", x=0.01, ha="left", fontweight="bold")
fig.tight_layout()
show(fig, "10_partial_dependence")

What pushes the probability up or down (logistic regression only). Coefficients are per standard deviation of each input, so they can be compared with each other. `log_area` is a control for how the labels are made, not a cause of flooding; see `riskengine/ml/README.md`.

In [ ]:
pd.Series(metrics.get("standardised_coefficients", {}), name="coefficient").sort_values().to_frame()

## 6. Save the run to Google Drive

Everything needed to trace or repeat this run goes into its own folder: the model, its metrics, the charts saved above, the training log, the installed packages and a short record of the run.

In [ ]:
import platform
import sys

shutil.copy(train.ARTIFACT_PATH, RUN_DIR / "flood_model.joblib")
shutil.copy(train.METRICS_PATH, RUN_DIR / "metrics.json")
(RUN_DIR / "environment.txt").write_text(
    subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True).stdout
)
(RUN_DIR / "run.json").write_text(json.dumps({
    "run_id": RUN_ID,
    "commit": COMMIT,
    "bundle": BUNDLE.name,
    "training_set_sha256": DATASET_SHA256,
    "model_version": artifact["version"],
    "python": platform.python_version(),
    "machine": platform.processor() or platform.machine(),
    "cores": os.cpu_count(),
    "seconds": TRAIN_SECONDS,
}, indent=2) + "\n")
drive.flush_and_unmount()  # wait until Drive has every file
print("Saved to", RUN_DIR)

## 7. Bring the run back

Copy `flood_model.joblib` and `metrics.json` from the run folder into `riskengine/ml/artifacts/` in the repository, replacing what is there, and the charts into `docs/figures/`. With Google Drive for desktop, wait for the sync to finish first. Then:

```
docker compose restart web
docker compose run --rm web python manage.py score_regions
docker compose run --rm web python manage.py flood_outlook
docker compose run --rm web python manage.py flood_outlook --level sub_county
```

Commit this notebook with its outputs as well, so the recorded code matches the run.

Without Google Drive for desktop, set `DOWNLOAD = True` below to download the two files through the browser instead.

In [ ]:
DOWNLOAD = False
if DOWNLOAD:
    from google.colab import files

    files.download(str(train.ARTIFACT_PATH))
    files.download(str(train.METRICS_PATH))